# Erdős Problem #30 (Option 2) — Deep Barrier & Higher-Order Hierarchy Cloud Notebook

Google AI Pro の **Google Colab Compute Units（推奨: CPU ハイメモリ）** を使って、Erdős #30 の $N^{1/4}$ 障壁突破研究と Lean 4 機械検証をクラウド上で再現・拡張するための専用ノートブックです。

含まれる実行パイプライン：
1. **Lean 4 機械検証**: `verify_gate.sh` による全形式証明定理（`sidon_sub_094295`, `shor_rlt_sdp1_lift`, `sdp1_sub_094295`, `bose_periodic_fourier_flatness`, `bose_3ap_free_superset_bound` 等）のクラウド検証
2. **Periodic Bose + Poisson 障壁 & $\theta$-整数性相転移**: 大規模 $N$（最大 $p = 1009, N > 10^6$）での点ごと自己相関検証
3. **Degree-1 vs Degree-2 Lasserre (SoS) 階層実験**: `SDP_1(N)` + 3-AP カット vs `Lasserre-2(N)`（4-tuple Sidon イデアル）の厳密凸 SDP 比較（ハイメモリ CPU で $N$ をさらに拡大可能）


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nproc && free -h


In [ ]:
import os
REPO_URL = "https://github.com/yoshi0703/ProofRush.git"
WORK_DIR = "/content/ProofRush"
if not os.path.exists(WORK_DIR):
    !git clone {REPO_URL} {WORK_DIR}
else:
    !git -C {WORK_DIR} pull --ff-only || true
%cd {WORK_DIR}
!pip -q install numpy scipy cvxpy clarabel z3-solver python-sat


In [ ]:
# 1. Periodic Bose + Poisson Barrier & Exact SDP_1 / Lasserre-2 Hierarchy Verification
!python3 solved/Erdos030/option2_hierarchy_experiments.py


In [ ]:
%%bash
# 2. Lean 4 + Mathlib Precompiled Cache Setup & Full verify_gate.sh Audit
set -euo pipefail
export PATH="$HOME/.elan/bin:$PATH"
if ! command -v elan >/dev/null 2>&1; then
  curl -sSf https://elan.lean-lang.org/elan-init.sh | sh -s -- -y --default-toolchain none
fi
cd /content/ProofRush/Formal
lake exe cache get
lake build Problems.Erdos030.Statement Problems.Erdos030.Solution
cd /content/ProofRush
mkdir -p state/locks
printf '%s\n' '3b2f86d3ed3e5c57f709a36adbe1ebbaa84252bae4b8b4a0b25333f3d0dba9f8' > state/locks/Erdos030.sha256
echo "[OK] Ready for Lean 4 formal verification on Colab!"
